In [ ]:
import torch
import os
from torch.utils.data import DataLoader
from torchvision.models import vgg19, VGG19_Weights
from tqdm import tqdm
from torchvision.utils import save_image
from IPython.display import Image as IPythonImage, display

from networks import Generator, Discriminator
from dataset import SRDataset
from losses import GLoss, DLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

In [ ]:
dataset = SRDataset(hr_dir="div2k\DIV2K_train_HR", patch_size=256, scale=4)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0, pin_memory=True)

G = Generator(64,23,32,5,4).to(device)
D = Discriminator().to(device)

vgg = vgg19(weights=VGG19_Weights.DEFAULT).features
feature_extractor = torch.nn.Sequential(*list(vgg.children())[:35]).to(device)
for param in feature_extractor.parameters():
    param.requires_grad = False

In [ ]:
G.load_state_dict(torch.load("generator_psnr.pth"))
print("Stage 1 weights loaded successfully! Starting GAN Training...")

optimizer_G = torch.optim.Adam(G.parameters(), lr=1e-5, betas=(0.9, 0.999))
optimizer_D = torch.optim.Adam(D.parameters(), lr=1e-5, betas=(0.9, 0.999))

L1 = torch.nn.L1Loss()
BCEloss = torch.nn.BCEWithLogitsLoss()

os.makedirs("training_progress", exist_ok=True)
epochs = 100

for epoch in range(epochs):
    loop = tqdm(dataloader, desc=f"Epoch [{epoch+1}/{epochs}]")
    
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0

    for lr, hr in loop:
        torch.cuda.empty_cache()
        lr = lr.to(device)
        hr = hr.to(device)

        sr = G(lr)

        optimizer_D.zero_grad()
        D_loss = DLoss(D, sr, hr, BCEloss)
        D_loss.backward()
        optimizer_D.step()

        optimizer_G.zero_grad()
        for p in D.parameters():
            p.requires_grad = False

        sr_feature_map = D(sr)
        
        G_loss = GLoss(sr, hr, sr_feature_map, feature_extractor, L1, BCEloss)

        G_loss.backward()
        optimizer_G.step()

        for p in D.parameters():
            p.requires_grad = True
            
        epoch_d_loss += D_loss.item()
        epoch_g_loss += G_loss.item()
        loop.set_postfix(D_loss=D_loss.item(), G_loss=G_loss.item())
        
    avg_d_loss = epoch_d_loss / len(dataloader)
    avg_g_loss = epoch_g_loss / len(dataloader)
    print(f"End of Epoch {epoch+1} | Avg D Loss: {avg_d_loss:.4f} | Avg G Loss: {avg_g_loss:.4f}")
    
    with torch.no_grad():
        lr_upscaled = torch.nn.functional.interpolate(lr, size=hr.shape[2:], mode='bicubic')
        comparison_grid = torch.cat([lr_upscaled[:2], sr[:2], hr[:2]], dim=0)
        
        img_path = f"training_progress/epoch_{epoch+1}.png"
        save_image(comparison_grid, img_path, nrow=2, normalize=True)
        
        print(f"Visualizing Stage 2 - Epoch {epoch+1}:")
        display(IPythonImage(filename=img_path))
        
    if (epoch + 1) % 10 == 0:
        torch.save(G.state_dict(), f"G_epoch_{epoch+1}.pth")
        print(f"Saved Generator weights for Epoch {epoch+1}")